In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()
os.chdir("..")
os.getcwd()

In [ ]:
import itertools
import os

import hydra
import lightning as pl
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn.functional as F
from lightning import LightningDataModule, Trainer

if os.environ.get("TOKENIZERS_PARALLELISM") is None:
    os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
from hydra import compose, initialize
from hydra.core.hydra_config import HydraConfig

with initialize(version_base="1.3", config_path="../configs"):
    cfg = compose(
        config_name="train.yaml", return_hydra_config=True, overrides=["experiment=pycharm"]
    )

HydraConfig.instance().set_config(cfg)

In [ ]:
datamodule: LightningDataModule = hydra.utils.instantiate(cfg.data)
datamodule.setup()

trainer: Trainer = pl.Trainer(accelerator="mps", max_epochs=1)
trainer.datamodule = datamodule

# Get model
model = hydra.utils.instantiate(cfg.model)
model.to("mps")
model.trainer = trainer
model.setup("test")

is_llm = "LLM" in cfg.model.text_encoder._target_

In [ ]:
captions = [c["concept_caption"] for c in datamodule.caption_builder.__dict__["concepts"]]
b = {"text": captions}
text_feats = model.text_encoder(b, "train")

In [ ]:
text_feats = F.normalize(text_feats)
cos_sim = text_feats @ text_feats.T
if is_llm:
    cos_sim = cos_sim.detach().cpu().to(torch.float32).numpy()
else:
    cos_sim = cos_sim.detach().cpu().detach().numpy()

In [ ]:
fig = plt.figure(figsize=(16, 8))
gs = fig.add_gridspec(1, 2, width_ratios=[1.5, 1])

ax1 = fig.add_subplot(gs[0])
short_labels = [f"C{i}" for i in range(len(captions))]

sns.heatmap(
    cos_sim,
    xticklabels=short_labels,
    yticklabels=short_labels,
    ax=ax1,
    cmap="viridis",
    # vmin=-1, vmax=1,
    cbar_kws={"label": "Cosine Similarity"},
)

ax1.set_title(
    f"Cosine Similarity Matrix For Concepts With {'LLM' if is_llm else 'CLIP'} Text Encoder"
)

ax2 = fig.add_subplot(gs[1])
ax2.axis("off")

legend_text = "\n".join([f"C{i}: {cap}" for i, cap in enumerate(captions)])
ax2.text(0, 1, legend_text, fontsize=10, va="top", ha="left", wrap=True)

plt.tight_layout()
plt.show()

In [ ]:
dataloader = datamodule.train_dataloader()
dataloader_cycle = itertools.cycle(dataloader)

all_captions = []
all_embeddings = []
total_samples = 0
max_seq_len = 0
target_samples = 10000

In [ ]:
for batch in dataloader_cycle:
    with torch.no_grad():
        model.eval()
        if is_llm:
            text_input = batch.get("text")
            text_feats = model.text_encoder(batch, "train")
        else:
            text_input = batch.get("text")
        text_tokens = model.text_encoder.processor(
            text=text_input,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=None,
        )
        if text_tokens.input_ids.shape[-1] <= 77:
            device = "mps"
            text_tokens = {k: v.to(device) for k, v in text_tokens.items()}
            text_embeds = model.text_encoder.model.get_text_features(**text_tokens)
            if model.text_encoder.projector is not None:
                text_embeds = model.text_encoder.projector(text_embeds)
            if model.text_encoder.extra_projector is not None:
                text_embeds = model.text_encoder.extra_projector(text_embeds)
        else:
            for i, t in enumerate(text_input):
                print(t, len(text_tokens["input_ids"][i]))

            if len(text_tokens["input_ids"][i]) > max_seq_len:
                max_seq_len = len(text_tokens["input_ids"][i])
            break

    all_embeddings.append(text_feats.cpu())
    all_captions.extend(batch["text"])

    total_samples += text_feats.shape[0]
    print(total_samples)

    # Break only when we have enough
    if total_samples >= target_samples:
        break

# 3. Finalize
text_feats = torch.cat(all_embeddings, dim=0)[:target_samples]
captions = all_captions[:target_samples]

text_feats = F.normalize(text_feats)
cos_sim = text_feats @ text_feats.T

In [ ]:
captions_arr = np.array(captions)
identity_mask = captions_arr[:, None] == captions_arr[None, :]
lower_tri_mask = np.tril(np.ones(identity_mask.shape, dtype=bool), k=-1)

final_mask = (~identity_mask) & lower_tri_mask
if is_llm:
    unique_sims = cos_sim.detach().cpu().to(torch.float32).numpy()[final_mask]
else:
    unique_sims = cos_sim.detach().cpu().numpy()[final_mask]

row_idx, col_idx = np.where(final_mask)

# Plot
plt.figure(figsize=(8, 5))
sns.histplot(unique_sims, bins=50, kde=True)
plt.title(
    f"Distribution of Cosine Similarities \n(for random 10k training location captions with {'LLM' if is_llm else 'CLIP'} text encoder)"
)
plt.xlabel("Cosine Similarity")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Get the lower triangle
lower_tri_indices = np.tril_indices(cos_sim.shape[0], k=-1)

# Get indices of the unique pairs
row_idx, col_idx = lower_tri_indices

# Sort the unique similarities
sorted_indices = np.argsort(unique_sims)  # Ascending order

print("--- Highest Similarity Pairs ---")
for i in sorted_indices[-5:][::-1]:
    r, c = row_idx[i], col_idx[i]
    print(f"Score: {unique_sims[i]:.4f}")
    print(f"  - '{captions[r]}'")
    print(f"  - '{captions[c]}'\n")

print("--- Lowest Similarity Pairs ---")
for i in sorted_indices[:5]:
    r, c = row_idx[i], col_idx[i]
    print(f"Score: {unique_sims[i]:.4f}")
    print(f"  - '{captions[r]}'")
    print(f"  - '{captions[c]}'\n")